# **XGBoost Model**

**Install required packages**

In [130]:
!pip install optuna plotly xgboost --quiet

**Import Libraries**

In [131]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import optuna
import warnings
from datetime import datetime
from google.colab import files
import io

warnings.filterwarnings('ignore')
pio.renderers.default = 'colab'

**Upload and Load Data**

In [132]:
print("\n📁 Please upload your CSV file")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df_original = pd.read_csv(io.BytesIO(uploaded[file_name]))

print(f"\nFile loaded: {len(df_original)} rows")
print("\nFirst 5 rows of data:")
print(df_original.head())

print("\nDataset Info:")
print(df_original.info())


📁 Please upload your CSV file


Saving final data.csv to final data (9).csv

File loaded: 120 rows

First 5 rows of data:
   Year     Month  Arrivals  Peak_Season  Philippine_Holidays  \
0  2016   January     57498            0                    2   
1  2016  February     65097            0                    2   
2  2016     March     58104            0                    3   
3  2016     April     56530            0                    1   
4  2016       May     58776            0                    2   

   Top10Market_Holidays  Avg_HighTemp  Avg_LowTemp  Precipitation  \
0                    17            31           24           0.10   
1                    20            30           24           0.17   
2                     9            32           24           0.04   
3                     6            32           25           0.07   
4                    22            33           26           0.29   

   Inflation_Rate  is_December  is_Lockdown  
0             1.6            0            0  
1           

**Data Preprocessing & Cleaning**

In [133]:
df['Date'] = pd.to_datetime(df['Year'].astype(str) + ' ' + df['Month'], format='%Y %B')
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
print(f"Total records: {len(df)} months")

Date range: 2016-01-01 to 2025-12-01
Total records: 120 months


**Defining Evaluation Function**

In [134]:
def evaluate_model_simple(y_true_train, y_pred_train, y_true_test, y_pred_test, model_name="Model"):

    def calc(y_true, y_pred):
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
        accuracy = 100 - mape
        return mae, rmse, mape, accuracy

    train_mae, train_rmse, train_mape, train_acc = calc(y_true_train, y_pred_train)
    test_mae, test_rmse, test_mape, test_acc = calc(y_true_test, y_pred_test)

# Get last date for future predictions
last_date = df.index.max()
future_3m = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=3, freq='MS')
future_6m = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=6, freq='MS')
future_12m = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=12, freq='MS')

# **XGBoost Version 1.0**

**Prepare data for Version 1**

In [135]:
df_v1 = df.copy()

**Feature Engineering for Version 1.0**

In [136]:
#Time Features
df_v1['Year'] = df_v1.index.year
df_v1['Month_Num'] = df_v1.index.month

# Drop NaN values
df_v1 = df_v1.dropna()

# Features for V1
features_v1 = ['Peak_Season', 'Philippine_Holidays', 'Top10Market_Holidays']
X_v1 = df_v1[features_v1]
y_v1 = df_v1['Arrivals']

**Split Data into Training & Testing (80-20 split)**

In [137]:
split_point = int(len(X_v1) * 0.8)
X_train_v1, X_test_v1 = X_v1.iloc[:split_point], X_v1.iloc[split_point:]
y_train_v1, y_test_v1 = y_v1.iloc[:split_point], y_v1.iloc[split_point:]

print(f"\nVersion 1.0 Data Split:")
print(f"Training: {X_train_v1.index.min().date()} to {X_train_v1.index.max().date()} ({len(X_train_v1)} months)")
print(f"Testing: {X_test_v1.index.min().date()} to {X_test_v1.index.max().date()} ({len(X_test_v1)} months)")


Version 1.0 Data Split:
Training: 2016-01-01 to 2023-12-01 (96 months)
Testing: 2024-01-01 to 2025-12-01 (24 months)


**Train Version 1.0 Model**

In [138]:
model_v1 = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=2,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1
)

model_v1.fit(X_train_v1, y_train_v1)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=2,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=-1, num_parallel_tree=None, ...)

**Predictions on Test Data**

In [139]:
y_pred_train_v1 = model_v1.predict(X_train_v1)
y_pred_test_v1 = model_v1.predict(X_test_v1)
y_pred_train_v1_rounded = np.round(y_pred_train_v1).astype(int)
y_pred_test_v1_rounded = np.round(y_pred_test_v1).astype(int)
train_metrics_v1= calculate_test_metrics(y_train_v1, y_pred_train_v1_rounded)
test_metrics_v1 = calculate_test_metrics(y_test_v1, y_pred_test_v1_rounded)

print("\nPredicted vs Actual Values:")
print(pd.DataFrame({'Actual': y_test_v1.values, 'Predicted': y_pred_test_v1_rounded}).head(10))


Predicted vs Actual Values:
   Actual  Predicted
0   72264      40599
1   65976      73581
2   67741      64134
3   71858      42079
4   72783      20645
5   66344      48918
6   71180      52559
7   74329      50982
8   63784      45992
9   66896      59555


**Model Evaluation & Visualization**

In [140]:
print("=" * 80)
print("VERSION 1.0: Model Evaluation & Visualization")
print("=" * 80)

# 1.1 TEST PERFORMANCE
print("\n📈 TEST PERFORMANCE:")
print("-" * 40)
print(f"   MAE: {test_metrics_v1['mae']:,.0f} arrivals")
print(f"   RMSE: {test_metrics_v1['rmse']:,.0f} arrivals")
print(f"   MAPE: {test_metrics_v1['mape']:.2f}%")
print(f"   Accuracy: {test_metrics_v1['accuracy']:.2f}%")

# 1.2 TEST SET PREDICTION CHART
print("\n📊 TEST SET PREDICTION CHART:")
print("-" * 40)

fig_v1_test = go.Figure()
fig_v1_test.add_trace(go.Scatter(
    x=X_test_v1.index,
    y=y_test_v1,
    mode='lines+markers',
    name='Actual',
    line=dict(color='blue', width=2),
    marker=dict(size=8, color='blue')
))

# Add predicted values
fig_v1_test.add_trace(go.Scatter(
    x=X_test_v1.index,
    y=y_pred_test_v1_rounded,
    mode='lines+markers',
    name='Predicted',
    line=dict(color='red', width=2, dash='dash'),
    marker=dict(size=8, color='red', symbol='circle')
))

fig_v1_test.update_layout(
    title='V1.0 - Actual vs Predicted Arrivals',
    xaxis_title='Month',
    yaxis_title='Arrivals',
    hovermode='x unified',
    template='plotly_white',
    height=450,
    width=1000,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
)
fig_v1_test.update_xaxes(tickformat='%b %Y', tickangle=45)
fig_v1_test.show()

# 1.3 FUTURE PREDICTIONS
print("\n🔮 FUTURE PREDICTIONS:")
print("-" * 40)

# Create future features for V1
future_v1_3m = pd.DataFrame(index=future_3m)
future_v1_3m['Peak_Season'] = future_v1_3m.index.month.isin([8,12]).astype(int)
future_v1_3m['Philippine_Holidays'] = future_v1_3m.index.month.map({1:2,2:1,3:2,4:3,5:2,6:2,7:1,8:2,9:1,10:1,11:3,12:4})
future_v1_3m['Top10Market_Holidays'] = 15

future_v1_6m = pd.DataFrame(index=future_6m)
future_v1_6m['Peak_Season'] = future_v1_6m.index.month.isin([8,12]).astype(int)
future_v1_6m['Philippine_Holidays'] = future_v1_6m.index.month.map({1:2,2:1,3:2,4:3,5:2,6:2,7:1,8:2,9:1,10:1,11:3,12:4})
future_v1_6m['Top10Market_Holidays'] = 15

future_v1_12m = pd.DataFrame(index=future_12m)
future_v1_12m['Peak_Season'] = future_v1_12m.index.month.isin([8,12]).astype(int)
future_v1_12m['Philippine_Holidays'] = future_v1_12m.index.month.map({1:2,2:1,3:2,4:3,5:2,6:2,7:1,8:2,9:1,10:1,11:3,12:4})
future_v1_12m['Top10Market_Holidays'] = 15

# Make predictions
pred_v1_3m = model_v1.predict(future_v1_3m[features_v1])
pred_v1_6m = model_v1.predict(future_v1_6m[features_v1])
pred_v1_12m = model_v1.predict(future_v1_12m[features_v1])

pred_v1_3m_rounded = np.round(pred_v1_3m).astype(int)
pred_v1_6m_rounded = np.round(pred_v1_6m).astype(int)
pred_v1_12m_rounded = np.round(pred_v1_12m).astype(int)

# Display 3-month predictions with chart
print("\n📅 NEXT 3 MONTHS PREDICTIONS:")
for i, date in enumerate(future_3m):
    print(f"   {date.strftime('%b %Y')}: {pred_v1_3m_rounded[i]:,} arrivals ({pred_v1_3m_rounded[i]/1000:.1f}k)")

# 3-Month Chart
fig_v1_3m = go.Figure()
fig_v1_3m.add_trace(go.Scatter(
    x=future_3m,
    y=pred_v1_3m_rounded,
    mode='lines+markers',
    name='V1.0 Predictions',
    line=dict(color='red', width=3),
    marker=dict(size=10, color='red', symbol='circle')
))
fig_v1_3m.update_layout(
    title='V1.0 - Next 3 Months Predictions',
    xaxis_title='Month',
    yaxis_title='Predicted Arrivals',
    template='plotly_white',
    height=400,
    width=800
)
fig_v1_3m.update_xaxes(tickformat='%b %Y', tickangle=0)
fig_v1_3m.show()

# Display 6-month predictions with chart
print("\n📅 NEXT 6 MONTHS PREDICTIONS:")
for i, date in enumerate(future_6m):
    print(f"   {date.strftime('%b %Y')}: {pred_v1_6m_rounded[i]:,} arrivals ({pred_v1_6m_rounded[i]/1000:.1f}k)")

# 6-Month Chart
fig_v1_6m = go.Figure()
fig_v1_6m.add_trace(go.Scatter(
    x=future_6m,
    y=pred_v1_6m_rounded,
    mode='lines+markers',
    name='V1.0 Predictions',
    line=dict(color='red', width=3),
    marker=dict(size=8, color='red', symbol='circle')
))
fig_v1_6m.update_layout(
    title='V1.0 - Next 6 Months Predictions',
    xaxis_title='Month',
    yaxis_title='Predicted Arrivals',
    template='plotly_white',
    height=400,
    width=900
)
fig_v1_6m.update_xaxes(tickformat='%b %Y', tickangle=45)
fig_v1_6m.show()

# Display 12-month predictions with chart
print("\n📅 NEXT 12 MONTHS PREDICTIONS:")
for i, date in enumerate(future_12m):
    print(f"   {date.strftime('%b %Y')}: {pred_v1_12m_rounded[i]:,} arrivals ({pred_v1_12m_rounded[i]/1000:.1f}k)")

# 12-Month Chart
fig_v1_12m = go.Figure()
fig_v1_12m.add_trace(go.Scatter(
    x=future_12m,
    y=pred_v1_12m_rounded,
    mode='lines+markers',
    name='V1.0 Predictions',
    line=dict(color='red', width=3),
    marker=dict(size=6, color='red', symbol='circle')
))
fig_v1_12m.update_layout(
    title='V1.0 - Next 12 Months Predictions',
    xaxis_title='Month',
    yaxis_title='Predicted Arrivals',
    template='plotly_white',
    height=400,
    width=1000
)
fig_v1_12m.update_xaxes(tickformat='%b %Y', tickangle=45)
fig_v1_12m.show()

VERSION 1.0: Model Evaluation & Visualization

📈 TEST PERFORMANCE:
----------------------------------------
   MAE: 25,064 arrivals
   RMSE: 28,799 arrivals
   MAPE: 33.25%
   Accuracy: 66.75%

📊 TEST SET PREDICTION CHART:
----------------------------------------



🔮 FUTURE PREDICTIONS:
----------------------------------------

📅 NEXT 3 MONTHS PREDICTIONS:
   Jan 2026: 42,079 arrivals (42.1k)
   Feb 2026: 40,599 arrivals (40.6k)
   Mar 2026: 42,079 arrivals (42.1k)



📅 NEXT 6 MONTHS PREDICTIONS:
   Jan 2026: 42,079 arrivals (42.1k)
   Feb 2026: 40,599 arrivals (40.6k)
   Mar 2026: 42,079 arrivals (42.1k)
   Apr 2026: 56,630 arrivals (56.6k)
   May 2026: 42,079 arrivals (42.1k)
   Jun 2026: 42,079 arrivals (42.1k)



📅 NEXT 12 MONTHS PREDICTIONS:
   Jan 2026: 42,079 arrivals (42.1k)
   Feb 2026: 40,599 arrivals (40.6k)
   Mar 2026: 42,079 arrivals (42.1k)
   Apr 2026: 56,630 arrivals (56.6k)
   May 2026: 42,079 arrivals (42.1k)
   Jun 2026: 42,079 arrivals (42.1k)
   Jul 2026: 40,599 arrivals (40.6k)
   Aug 2026: 18,986 arrivals (19.0k)
   Sep 2026: 40,599 arrivals (40.6k)
   Oct 2026: 40,599 arrivals (40.6k)
   Nov 2026: 56,630 arrivals (56.6k)
   Dec 2026: 33,537 arrivals (33.5k)


**Overfitting Check**

In [143]:
print("\n⚠️ V1.0 OVERFITTING CHECK:")
r2_diff_v1 = train_metrics_v1['r2'] - test_metrics_v1['r2']
mae_ratio_v1 = test_metrics_v1['mae'] / train_metrics_v1['mae']
print(f"R² Difference: {r2_diff_v1:.4f}")
print(f"MAE Ratio: {mae_ratio_v1:.2f}x")


⚠️ V1.0 OVERFITTING CHECK:


KeyError: 'r2'

# **XGBoost Version 2.0**

**Prepare data for Version 2**

In [144]:
df_v2 = df.copy()

**Feature Engineering for Version 2.0**

In [145]:
#Time features
df_v2['Year'] = df_v2.index.year
df_v2['Month_Num'] = df_v2.index.month

# Drop NaN
df_v2 = df_v2.dropna()

# Features for V2
features_v2 = ['Peak_Season', 'Philippine_Holidays', 'Top10Market_Holidays',
               'Avg_HighTemp', 'Avg_LowTemp', 'Precipitation',
               'Inflation_Rate', 'is_December', 'is_Lockdown']

X_v2 = df_v2[features_v2]
y_v2 = df_v2['Arrivals']

**Split Data into Training & Testing (80-20 split)**

In [146]:
X_train_v2, X_test_v2 = X_v2.iloc[:split_point], X_v2.iloc[split_point:]
y_train_v2, y_test_v2 = y_v2.iloc[:split_point], y_v2.iloc[split_point:]
print(f"\n📊 V2 Data Split:")
print(f"Training: {X_train_v2.index.min().date()} to {X_train_v2.index.max().date()} ({len(X_train_v2)} months)")
print(f"Testing: {X_test_v2.index.min().date()} to {X_test_v2.index.max().date()} ({len(X_test_v2)} months)")


📊 V2 Data Split:
Training: 2016-01-01 to 2023-12-01 (96 months)
Testing: 2024-01-01 to 2025-12-01 (24 months)


**Train Version 2.0 Model**

In [147]:
model_v2 = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=2,
    learning_rate=0.05,
    reg_alpha=0.5,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

model_v2.fit(X_train_v2, y_train_v2)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=2,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=-1, num_parallel_tree=None, ...)

**Predictions on Test Data**

In [148]:
y_pred_train_v2 = model_v2.predict(X_train_v2)
y_pred_test_v2 = model_v2.predict(X_test_v2)
y_pred_train_v2_rounded = np.round(y_pred_train_v2).astype(int)
y_pred_test_v2_rounded = np.round(y_pred_test_v2).astype(int)
train_metrics_v2= calculate_test_metrics(y_train_v2, y_pred_train_v2_rounded)
test_metrics_v2 = calculate_test_metrics(y_test_v2, y_pred_test_v2_rounded)

print("\nPredicted vs Actual Values:")
print(pd.DataFrame({'Actual': y_test_v2.values, 'Predicted': y_pred_test_v2_rounded}).head(10))


Predicted vs Actual Values:
   Actual  Predicted
0   72264      78916
1   65976      94848
2   67741      78792
3   71858      93399
4   72783      60242
5   66344      41310
6   71180      43967
7   74329      30768
8   63784      19247
9   66896      34041


**Model Evaluation & Visualization**

In [149]:
print("\n" + "="*80)
print("📊 VERSION 2.0: Model Evaluation & Visualization")
print("="*80)

# 2.1 TEST PERFORMANCE
print("\n📈 TEST PERFORMANCE:")
print("-" * 40)
print(f"   MAE: {test_metrics_v2['mae']:,.0f} arrivals")
print(f"   RMSE: {test_metrics_v2['rmse']:,.0f} arrivals")
print(f"   MAPE: {test_metrics_v2['mape']:.2f}%")
print(f"   Accuracy: {test_metrics_v2['accuracy']:.2f}%")

# 2.2 TEST SET PREDICTION CHART
print("\n📊 TEST SET PREDICTION CHART:")
print("-" * 40)

fig_v2_test = go.Figure()
fig_v2_test.add_trace(go.Scatter(
    x=X_test_v2.index,
    y=y_test_v2,
    mode='lines+markers',
    name='Actual',
    line=dict(color='blue', width=2),
    marker=dict(size=8, color='blue')
))
fig_v2_test.add_trace(go.Scatter(
    x=X_test_v2.index,
    y=y_pred_test_v2_rounded,
    mode='lines+markers',
    name='Predicted',
    line=dict(color='red', width=2, dash='dash'),
    marker=dict(size=8, color='red', symbol='circle')
))
fig_v2_test.update_layout(
    title='V2.0 - Actual vs Predicted Arrivals',
    xaxis_title='Month',
    yaxis_title='Arrivals',
    hovermode='x unified',
    template='plotly_white',
    height=450,
    width=1000,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
)
fig_v2_test.update_xaxes(tickformat='%b %Y', tickangle=45)
fig_v2_test.show()

# 2.3 FUTURE PREDICTIONS
print("\n🔮 FUTURE PREDICTIONS:")
print("-" * 40)

# Weather maps
temp_high_map = {1:29,2:29,3:30,4:31,5:32,6:32,7:32,8:32,9:32,10:31,11:30,12:29}
temp_low_map = {1:25,2:25,3:25,4:26,5:26,6:26,7:26,8:26,9:26,10:26,11:25,12:25}
precip_map = {1:0.7,2:0.5,3:0.4,4:0.3,5:0.4,6:0.7,7:0.9,8:0.8,9:0.9,10:1.0,11:0.9,12:0.7}

# Create future features for V2
future_v2_3m = pd.DataFrame(index=future_3m)
future_v2_3m['Peak_Season'] = future_v2_3m.index.month.isin([8,12]).astype(int)
future_v2_3m['Philippine_Holidays'] = future_v2_3m.index.month.map({1:2,2:1,3:2,4:3,5:2,6:2,7:1,8:2,9:1,10:1,11:3,12:4})
future_v2_3m['Top10Market_Holidays'] = 15
future_v2_3m['Avg_HighTemp'] = future_v2_3m.index.month.map(temp_high_map)
future_v2_3m['Avg_LowTemp'] = future_v2_3m.index.month.map(temp_low_map)
future_v2_3m['Precipitation'] = future_v2_3m.index.month.map(precip_map)
future_v2_3m['Inflation_Rate'] = 3.0
future_v2_3m['is_December'] = (future_v2_3m.index.month == 12).astype(int)
future_v2_3m['is_Lockdown'] = 0

future_v2_6m = pd.DataFrame(index=future_6m)
future_v2_6m['Peak_Season'] = future_v2_6m.index.month.isin([8,12]).astype(int)
future_v2_6m['Philippine_Holidays'] = future_v2_6m.index.month.map({1:2,2:1,3:2,4:3,5:2,6:2,7:1,8:2,9:1,10:1,11:3,12:4})
future_v2_6m['Top10Market_Holidays'] = 15
future_v2_6m['Avg_HighTemp'] = future_v2_6m.index.month.map(temp_high_map)
future_v2_6m['Avg_LowTemp'] = future_v2_6m.index.month.map(temp_low_map)
future_v2_6m['Precipitation'] = future_v2_6m.index.month.map(precip_map)
future_v2_6m['Inflation_Rate'] = 3.0
future_v2_6m['is_December'] = (future_v2_6m.index.month == 12).astype(int)
future_v2_6m['is_Lockdown'] = 0

future_v2_12m = pd.DataFrame(index=future_12m)
future_v2_12m['Peak_Season'] = future_v2_12m.index.month.isin([8,12]).astype(int)
future_v2_12m['Philippine_Holidays'] = future_v2_12m.index.month.map({1:2,2:1,3:2,4:3,5:2,6:2,7:1,8:2,9:1,10:1,11:3,12:4})
future_v2_12m['Top10Market_Holidays'] = 15
future_v2_12m['Avg_HighTemp'] = future_v2_12m.index.month.map(temp_high_map)
future_v2_12m['Avg_LowTemp'] = future_v2_12m.index.month.map(temp_low_map)
future_v2_12m['Precipitation'] = future_v2_12m.index.month.map(precip_map)
future_v2_12m['Inflation_Rate'] = 3.0
future_v2_12m['is_December'] = (future_v2_12m.index.month == 12).astype(int)
future_v2_12m['is_Lockdown'] = 0

# Make predictions
pred_v2_3m = model_v2.predict(future_v2_3m[features_v2])
pred_v2_6m = model_v2.predict(future_v2_6m[features_v2])
pred_v2_12m = model_v2.predict(future_v2_12m[features_v2])

pred_v2_3m_rounded = np.round(pred_v2_3m).astype(int)
pred_v2_6m_rounded = np.round(pred_v2_6m).astype(int)
pred_v2_12m_rounded = np.round(pred_v2_12m).astype(int)

# Display 3-month predictions with chart
print("\n📅 NEXT 3 MONTHS PREDICTIONS:")
for i, date in enumerate(future_3m):
    print(f"   {date.strftime('%b %Y')}: {pred_v2_3m_rounded[i]:,} arrivals ({pred_v2_3m_rounded[i]/1000:.1f}k)")

fig_v2_3m = go.Figure()
fig_v2_3m.add_trace(go.Scatter(
    x=future_3m,
    y=pred_v2_3m_rounded,
    mode='lines+markers',
    name='V2.0 Predictions',
    line=dict(color='red', width=3),
    marker=dict(size=10, color='red', symbol='circle')
))
fig_v2_3m.update_layout(
    title='V2.0 - Next 3 Months Predictions',
    xaxis_title='Month',
    yaxis_title='Predicted Arrivals',
    template='plotly_white',
    height=400,
    width=800
)
fig_v2_3m.update_xaxes(tickformat='%b %Y', tickangle=0)
fig_v2_3m.show()

# Display 6-month predictions with chart
print("\n📅 NEXT 6 MONTHS PREDICTIONS:")
for i, date in enumerate(future_6m):
    print(f"   {date.strftime('%b %Y')}: {pred_v2_6m_rounded[i]:,} arrivals ({pred_v2_6m_rounded[i]/1000:.1f}k)")

fig_v2_6m = go.Figure()
fig_v2_6m.add_trace(go.Scatter(
    x=future_6m,
    y=pred_v2_6m_rounded,
    mode='lines+markers',
    name='V2.0 Predictions',
    line=dict(color='red', width=3),
    marker=dict(size=8, color='red', symbol='circle')
))
fig_v2_6m.update_layout(
    title='V2.0 - Next 6 Months Predictions',
    xaxis_title='Month',
    yaxis_title='Predicted Arrivals',
    template='plotly_white',
    height=400,
    width=900
)
fig_v2_6m.update_xaxes(tickformat='%b %Y', tickangle=45)
fig_v2_6m.show()

# Display 12-month predictions with chart
print("\n📅 NEXT 12 MONTHS PREDICTIONS:")
for i, date in enumerate(future_12m):
    print(f"   {date.strftime('%b %Y')}: {pred_v2_12m_rounded[i]:,} arrivals ({pred_v2_12m_rounded[i]/1000:.1f}k)")

fig_v2_12m = go.Figure()
fig_v2_12m.add_trace(go.Scatter(
    x=future_12m,
    y=pred_v2_12m_rounded,
    mode='lines+markers',
    name='V2.0 Predictions',
    line=dict(color='red', width=3),
    marker=dict(size=6, color='red', symbol='circle')
))
fig_v2_12m.update_layout(
    title='V2.0 - Next 12 Months Predictions',
    xaxis_title='Month',
    yaxis_title='Predicted Arrivals',
    template='plotly_white',
    height=400,
    width=1000
)
fig_v2_12m.update_xaxes(tickformat='%b %Y', tickangle=45)
fig_v2_12m.show()


📊 VERSION 2.0: Model Evaluation & Visualization

📈 TEST PERFORMANCE:
----------------------------------------
   MAE: 29,147 arrivals
   RMSE: 32,072 arrivals
   MAPE: 38.89%
   Accuracy: 61.11%

📊 TEST SET PREDICTION CHART:
----------------------------------------



🔮 FUTURE PREDICTIONS:
----------------------------------------

📅 NEXT 3 MONTHS PREDICTIONS:
   Jan 2026: 29,104 arrivals (29.1k)
   Feb 2026: 32,328 arrivals (32.3k)
   Mar 2026: 43,572 arrivals (43.6k)



📅 NEXT 6 MONTHS PREDICTIONS:
   Jan 2026: 29,104 arrivals (29.1k)
   Feb 2026: 32,328 arrivals (32.3k)
   Mar 2026: 43,572 arrivals (43.6k)
   Apr 2026: 52,121 arrivals (52.1k)
   May 2026: 49,430 arrivals (49.4k)
   Jun 2026: 34,962 arrivals (35.0k)



📅 NEXT 12 MONTHS PREDICTIONS:
   Jan 2026: 29,104 arrivals (29.1k)
   Feb 2026: 32,328 arrivals (32.3k)
   Mar 2026: 43,572 arrivals (43.6k)
   Apr 2026: 52,121 arrivals (52.1k)
   May 2026: 49,430 arrivals (49.4k)
   Jun 2026: 34,962 arrivals (35.0k)
   Jul 2026: 19,493 arrivals (19.5k)
   Aug 2026: 35,735 arrivals (35.7k)
   Sep 2026: 19,493 arrivals (19.5k)
   Oct 2026: 20,638 arrivals (20.6k)
   Nov 2026: 11,913 arrivals (11.9k)
   Dec 2026: 26,339 arrivals (26.3k)


**Overfitting Check**

In [150]:
print("\n⚠️ V2.0 OVERFITTING CHECK:")
r2_diff_v2 = train_metrics_v2['r2'] - test_metrics_v2['r2']
mae_ratio_v2 = test_metrics_v2['mae'] / train_metrics_v2['mae']
print(f"R² Difference: {r2_diff_v2:.4f}")
print(f"MAE Ratio: {mae_ratio_v2:.2f}x")


⚠️ V2.0 OVERFITTING CHECK:


KeyError: 'r2'

# **XGBoost Version 3.0**

**Prepare data for Version 3.0**

In [151]:
df_v3 = df.copy()

**Feature Engineering for Version 3.0**

In [152]:
#Lag Features
df_v3['Lag_1'] = df_v3['Arrivals'].shift(1)
df_v3['Lag_2'] = df_v3['Arrivals'].shift(2)
df_v3['Rolling_Mean_3m'] = df_v3['Arrivals'].rolling(window=3, min_periods=1).mean()

# Time Features
df_v3['Year'] = df_v3.index.year
df_v3['Month_Num'] = df_v3.index.month

# Growth Rate
df_v3['Growth_Rate'] = df_v3['Arrivals'].pct_change() * 100
df_v3['Sudden_Increase'] = (df_v3['Growth_Rate'] > df_v3['Growth_Rate'].quantile(0.75)).astype(int)

# Drop NaN from Lag Features
df_v3 = df_v3.dropna()

# Features for V3
features_v3 = ['Peak_Season', 'Philippine_Holidays', 'Top10Market_Holidays',
               'Avg_HighTemp', 'Avg_LowTemp', 'Precipitation',
               'Lag_1', 'Lag_2', 'Rolling_Mean_3m',
               'Year', 'Month_Num', 'Growth_Rate', 'Sudden_Increase',
               'Inflation_Rate', 'is_December', 'is_Lockdown']

X_v3 = df_v3[features_v3]
y_v3 = df_v3['Arrivals']

**Split Data into Training & Testing (80-20 split)**

In [153]:
X_train_v3, X_test_v3 = X_v3.iloc[:split_point], X_v3.iloc[split_point:]
y_train_v3, y_test_v3 = y_v3.iloc[:split_point], y_v3.iloc[split_point:]

print(f"\n📊 V3 Data Split:")
print(f"Training: {X_train_v3.index.min().date()} to {X_train_v3.index.max().date()} ({len(X_train_v3)} months)")
print(f"Testing: {X_test_v3.index.min().date()} to {X_test_v3.index.max().date()} ({len(X_test_v3)} months)")



📊 V3 Data Split:
Training: 2016-03-01 to 2024-02-01 (96 months)
Testing: 2024-03-01 to 2025-12-01 (22 months)


**Simple Optuna Optimization**

In [154]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'max_depth': trial.suggest_int('max_depth', 2, 5),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 2.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 2.0),
    }

    model = xgb.XGBRegressor(**params, random_state=42, n_jobs=-1)
    model.fit(X_train_v3, y_train_v3)
    y_pred = model.predict(X_test_v3)
    return mean_absolute_error(y_test_v3, y_pred)

print("\n🎯 Running Optuna optimization (20 trials)...")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

best_params = study.best_params
print(f"\n✅ Best parameters found: {best_params}")

[I 2026-03-17 06:18:24,994] A new study created in memory with name: no-name-44805554-aec4-4dd3-aaf9-afeec3a6524c



🎯 Running Optuna optimization (20 trials)...


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-03-17 06:18:25,182] Trial 0 finished with value: 3075.000732421875 and parameters: {'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.010901449192594299, 'subsample': 0.8135945508253779, 'colsample_bytree': 0.8266209843019524, 'reg_alpha': 0.2569409939313645, 'reg_lambda': 1.9114007486488687}. Best is trial 0 with value: 3075.000732421875.
[I 2026-03-17 06:18:25,321] Trial 1 finished with value: 3241.228759765625 and parameters: {'n_estimators': 500, 'max_depth': 2, 'learning_rate': 0.010755702195230919, 'subsample': 0.8134371865475993, 'colsample_bytree': 0.8544267451531562, 'reg_alpha': 0.13491050940170643, 'reg_lambda': 0.6086303935249556}. Best is trial 0 with value: 3075.000732421875.
[I 2026-03-17 06:18:25,437] Trial 2 finished with value: 3081.09228515625 and parameters: {'n_estimators': 350, 'max_depth': 2, 'learning_rate': 0.015752999021081692, 'subsample': 0.6092160359250147, 'colsample_bytree': 0.8593370516610359, 'reg_alpha': 1.727634319648016, 'reg_lambda': 0

**Train Version 3.0 Model**

In [155]:
model_v3 = xgb.XGBRegressor(**best_params, random_state=42, n_jobs=-1)
model_v3.fit(X_train_v3, y_train_v3)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8995708310976516, device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, feature_weights=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.036306443636682215,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=2, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=400, n_jobs=-1,
             num_parallel_tree=None, ...)

**Predictions on Test Data**

In [156]:
y_pred_train_v3 = model_v3.predict(X_train_v3)
y_pred_test_v3 = model_v3.predict(X_test_v3)
y_pred_train_v3_rounded = np.round(y_pred_train_v3).astype(int)
y_pred_test_v3_rounded = np.round(y_pred_test_v3).astype(int)
train_metrics_v3= calculate_test_metrics(y_train_v3, y_pred_train_v3_rounded)
test_metrics_v3 = calculate_test_metrics(y_test_v3, y_pred_test_v3_rounded)

print("\nPredicted vs Actual Values:")
print(pd.DataFrame({'Actual': y_test_v3.values, 'Predicted': y_pred_test_v3_rounded}).head(10))


Predicted vs Actual Values:
   Actual  Predicted
0   67741      69307
1   71858      71173
2   72783      68846
3   66344      66595
4   71180      70572
5   74329      69978
6   63784      62454
7   66896      66512
8   79680      76236
9   84679      80801


**Model Evaluation & Visualization**

In [157]:
print("\n" + "="*80)
print("📊 VERSION 3.0: Model Evaluation & Visualization")
print("="*80)

# 3.1 TEST PERFORMANCE
print("\n📈 TEST PERFORMANCE:")
print("-" * 40)
print(f"   MAE: {test_metrics_v3['mae']:,.0f} arrivals")
print(f"   RMSE: {test_metrics_v3['rmse']:,.0f} arrivals")
print(f"   MAPE: {test_metrics_v3['mape']:.2f}%")
print(f"   Accuracy: {test_metrics_v3['accuracy']:.2f}%")

# 3.2 TEST SET PREDICTION CHART
print("\n📊 TEST SET PREDICTION CHART:")
print("-" * 40)

fig_v3_test = go.Figure()
fig_v3_test.add_trace(go.Scatter(
    x=X_test_v3.index,
    y=y_test_v3,
    mode='lines+markers',
    name='Actual',
    line=dict(color='blue', width=2),
    marker=dict(size=8, color='blue')
))
fig_v3_test.add_trace(go.Scatter(
    x=X_test_v3.index,
    y=y_pred_test_v3_rounded,
    mode='lines+markers',
    name='Predicted',
    line=dict(color='red', width=2, dash='dash'),
    marker=dict(size=8, color='red', symbol='circle')
))
fig_v3_test.update_layout(
    title='V3.0 - Actual vs Predicted Arrivals',
    xaxis_title='Month',
    yaxis_title='Arrivals',
    hovermode='x unified',
    template='plotly_white',
    height=450,
    width=1000,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
)
fig_v3_test.update_xaxes(tickformat='%b %Y', tickangle=45)
fig_v3_test.show()

# 3.3 FUTURE PREDICTIONS
print("\n🔮 FUTURE PREDICTIONS:")
print("-" * 40)

# Get last known values for lag features
last_values = df_v3['Arrivals'].values[-3:] if 'df_v3' in dir() else df['Arrivals'].values[-3:]

# Create future features for V3
future_v3_3m = pd.DataFrame(index=future_3m)
future_v3_3m['Peak_Season'] = future_v3_3m.index.month.isin([8,12]).astype(int)
future_v3_3m['Philippine_Holidays'] = future_v3_3m.index.month.map({1:2,2:1,3:2,4:3,5:2,6:2,7:1,8:2,9:1,10:1,11:3,12:4})
future_v3_3m['Top10Market_Holidays'] = 15
future_v3_3m['Avg_HighTemp'] = future_v3_3m.index.month.map(temp_high_map)
future_v3_3m['Avg_LowTemp'] = future_v3_3m.index.month.map(temp_low_map)
future_v3_3m['Precipitation'] = future_v3_3m.index.month.map(precip_map)
future_v3_3m['Inflation_Rate'] = 3.0
future_v3_3m['is_December'] = (future_v3_3m.index.month == 12).astype(int)
future_v3_3m['is_Lockdown'] = 0
future_v3_3m['Year'] = future_v3_3m.index.year
future_v3_3m['Month_Num'] = future_v3_3m.index.month
future_v3_3m['Lag_1'] = last_values[-1]
future_v3_3m['Lag_2'] = last_values[-2]
future_v3_3m['Rolling_Mean_3m'] = last_values.mean()
future_v3_3m['Growth_Rate'] = ((last_values[-1] - last_values[-2]) / last_values[-2] * 100)
future_v3_3m['Sudden_Increase'] = 0

future_v3_6m = pd.DataFrame(index=future_6m)
future_v3_6m['Peak_Season'] = future_v3_6m.index.month.isin([8,12]).astype(int)
future_v3_6m['Philippine_Holidays'] = future_v3_6m.index.month.map({1:2,2:1,3:2,4:3,5:2,6:2,7:1,8:2,9:1,10:1,11:3,12:4})
future_v3_6m['Top10Market_Holidays'] = 15
future_v3_6m['Avg_HighTemp'] = future_v3_6m.index.month.map(temp_high_map)
future_v3_6m['Avg_LowTemp'] = future_v3_6m.index.month.map(temp_low_map)
future_v3_6m['Precipitation'] = future_v3_6m.index.month.map(precip_map)
future_v3_6m['Inflation_Rate'] = 3.0
future_v3_6m['is_December'] = (future_v3_6m.index.month == 12).astype(int)
future_v3_6m['is_Lockdown'] = 0
future_v3_6m['Year'] = future_v3_6m.index.year
future_v3_6m['Month_Num'] = future_v3_6m.index.month
future_v3_6m['Lag_1'] = last_values[-1]
future_v3_6m['Lag_2'] = last_values[-2]
future_v3_6m['Rolling_Mean_3m'] = last_values.mean()
future_v3_6m['Growth_Rate'] = ((last_values[-1] - last_values[-2]) / last_values[-2] * 100)
future_v3_6m['Sudden_Increase'] = 0

future_v3_12m = pd.DataFrame(index=future_12m)
for df_future in [future_v3_12m]:
    df_future['Peak_Season'] = df_future.index.month.isin([8,12]).astype(int)
    df_future['Philippine_Holidays'] = df_future.index.month.map({1:2,2:1,3:2,4:3,5:2,6:2,7:1,8:2,9:1,10:1,11:3,12:4})
    df_future['Top10Market_Holidays'] = 15
    df_future['Avg_HighTemp'] = df_future.index.month.map(temp_high_map)
    df_future['Avg_LowTemp'] = df_future.index.month.map(temp_low_map)
    df_future['Precipitation'] = df_future.index.month.map(precip_map)
    df_future['Inflation_Rate'] = 3.0
    df_future['is_December'] = (df_future.index.month == 12).astype(int)
    df_future['is_Lockdown'] = 0
    df_future['Year'] = df_future.index.year
    df_future['Month_Num'] = df_future.index.month
    df_future['Lag_1'] = last_values[-1]
    df_future['Lag_2'] = last_values[-2]
    df_future['Rolling_Mean_3m'] = last_values.mean()
    df_future['Growth_Rate'] = ((last_values[-1] - last_values[-2]) / last_values[-2] * 100)
    df_future['Sudden_Increase'] = 0

# Make predictions
pred_v3_3m = model_v3.predict(future_v3_3m[features_v3])
pred_v3_6m = model_v3.predict(future_v3_6m[features_v3])
pred_v3_12m = model_v3.predict(future_v3_12m[features_v3])

pred_v3_3m_rounded = np.round(pred_v3_3m).astype(int)
pred_v3_6m_rounded = np.round(pred_v3_6m).astype(int)
pred_v3_12m_rounded = np.round(pred_v3_12m).astype(int)

# Display 3-month predictions with chart
print("\n📅 NEXT 3 MONTHS PREDICTIONS:")
for i, date in enumerate(future_3m):
    print(f"   {date.strftime('%b %Y')}: {pred_v3_3m_rounded[i]:,} arrivals ({pred_v3_3m_rounded[i]/1000:.1f}k)")

fig_v3_3m = go.Figure()
fig_v3_3m.add_trace(go.Scatter(
    x=future_3m,
    y=pred_v3_3m_rounded,
    mode='lines+markers',
    name='V3.0 Predictions',
    line=dict(color='red', width=3),
    marker=dict(size=10, color='red', symbol='circle')
))
fig_v3_3m.update_layout(
    title='V3.0 - Next 3 Months Predictions',
    xaxis_title='Month',
    yaxis_title='Predicted Arrivals',
    template='plotly_white',
    height=400,
    width=800
)
fig_v3_3m.update_xaxes(tickformat='%b %Y', tickangle=0)
fig_v3_3m.show()

# Display 6-month predictions with chart
print("\n📅 NEXT 6 MONTHS PREDICTIONS:")
for i, date in enumerate(future_6m):
    print(f"   {date.strftime('%b %Y')}: {pred_v3_6m_rounded[i]:,} arrivals ({pred_v3_6m_rounded[i]/1000:.1f}k)")

fig_v3_6m = go.Figure()
fig_v3_6m.add_trace(go.Scatter(
    x=future_6m,
    y=pred_v3_6m_rounded,
    mode='lines+markers',
    name='V3.0 Predictions',
    line=dict(color='red', width=3),
    marker=dict(size=8, color='red', symbol='circle')
))
fig_v3_6m.update_layout(
    title='V3.0 - Next 6 Months Predictions',
    xaxis_title='Month',
    yaxis_title='Predicted Arrivals',
    template='plotly_white',
    height=400,
    width=900
)
fig_v3_6m.update_xaxes(tickformat='%b %Y', tickangle=45)
fig_v3_6m.show()

# Display 12-month predictions with chart
print("\n📅 NEXT 12 MONTHS PREDICTIONS:")
for i, date in enumerate(future_12m):
    print(f"   {date.strftime('%b %Y')}: {pred_v3_12m_rounded[i]:,} arrivals ({pred_v3_12m_rounded[i]/1000:.1f}k)")

fig_v3_12m = go.Figure()
fig_v3_12m.add_trace(go.Scatter(
    x=future_12m,
    y=pred_v3_12m_rounded,
    mode='lines+markers',
    name='V3.0 Predictions',
    line=dict(color='red', width=3),
    marker=dict(size=6, color='red', symbol='circle')
))
fig_v3_12m.update_layout(
    title='V3.0 - Next 12 Months Predictions',
    xaxis_title='Month',
    yaxis_title='Predicted Arrivals',
    template='plotly_white',
    height=400,
    width=1000
)
fig_v3_12m.update_xaxes(tickformat='%b %Y', tickangle=45)
fig_v3_12m.show()


📊 VERSION 3.0: Model Evaluation & Visualization

📈 TEST PERFORMANCE:
----------------------------------------
   MAE: 2,295 arrivals
   RMSE: 3,129 arrivals
   MAPE: 2.91%
   Accuracy: 97.09%

📊 TEST SET PREDICTION CHART:
----------------------------------------



🔮 FUTURE PREDICTIONS:
----------------------------------------

📅 NEXT 3 MONTHS PREDICTIONS:
   Jan 2026: 75,201 arrivals (75.2k)
   Feb 2026: 74,022 arrivals (74.0k)
   Mar 2026: 73,650 arrivals (73.7k)



📅 NEXT 6 MONTHS PREDICTIONS:
   Jan 2026: 75,201 arrivals (75.2k)
   Feb 2026: 74,022 arrivals (74.0k)
   Mar 2026: 73,650 arrivals (73.7k)
   Apr 2026: 72,956 arrivals (73.0k)
   May 2026: 72,838 arrivals (72.8k)
   Jun 2026: 72,800 arrivals (72.8k)



📅 NEXT 12 MONTHS PREDICTIONS:
   Jan 2026: 75,201 arrivals (75.2k)
   Feb 2026: 74,022 arrivals (74.0k)
   Mar 2026: 73,650 arrivals (73.7k)
   Apr 2026: 72,956 arrivals (73.0k)
   May 2026: 72,838 arrivals (72.8k)
   Jun 2026: 72,800 arrivals (72.8k)
   Jul 2026: 72,779 arrivals (72.8k)
   Aug 2026: 72,739 arrivals (72.7k)
   Sep 2026: 72,611 arrivals (72.6k)
   Oct 2026: 72,797 arrivals (72.8k)
   Nov 2026: 74,177 arrivals (74.2k)
   Dec 2026: 74,668 arrivals (74.7k)


**Overfitting Check**

In [ ]:
print("\n⚠️ V3.0 OVERFITTING CHECK:")
r2_diff_v3 = train_metrics_v3['r2'] - test_metrics_v3['r2']
mae_ratio_v3 = test_metrics_v3['mae'] / train_metrics_v3['mae']
print(f"R² Difference: {r2_diff_v3:.4f}")
print(f"MAE Ratio: {mae_ratio_v3:.2f}x")

In [ ]:
#Model Comparison Dashboard
fig_dashboard = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        '<b>Mean Absolute Error (MAE)</b>',
        '<b>R² Score</b>',
        '<b>Test Accuracy (%)</b>',
        '<b>Overfitting: R² Difference</b>'
    ),
    specs=[
        [{"type": "bar"}, {"type": "bar"}],
        [{"type": "bar"}, {"type": "bar"}]
    ]
)

# MAE Subplot
fig_dashboard.add_trace(
    go.Bar(
        name='Train MAE',
        x=['V1', 'V2', 'V3'],
        y=[train_metrics_v1['mae'], train_metrics_v2['mae'], train_metrics_v3['mae']],
        marker_color='#1f77b4',
        legendgroup='train',
        text=[f"{train_metrics_v1['mae']:,.0f}", f"{train_metrics_v2['mae']:,.0f}", f"{train_metrics_v3['mae']:,.0f}"],
        textposition='inside',
        textfont=dict(size=10, color='white', family="Arial Black")
    ),
    row=1, col=1
)

fig_dashboard.add_trace(
    go.Bar(
        name='Test MAE',
        x=['V1', 'V2', 'V3'],
        y=[test_metrics_v1['mae'], test_metrics_v2['mae'], test_metrics_v3['mae']],
        marker_color='#ff7f0e',
        legendgroup='test',
        text=[f"{test_metrics_v1['mae']:,.0f}", f"{test_metrics_v2['mae']:,.0f}", f"{test_metrics_v3['mae']:,.0f}"],
        textposition='inside',
        textfont=dict(size=10, color='white', family="Arial Black")
    ),
    row=1, col=1
)

# R² Subplot
fig_dashboard.add_trace(
    go.Bar(
        name='Train R²',
        x=['V1', 'V2', 'V3'],
        y=[train_metrics_v1['r2'], train_metrics_v2['r2'], train_metrics_v3['r2']],
        marker_color='#1f77b4',
        showlegend=False,
        text=[f"{train_metrics_v1['r2']:.3f}", f"{train_metrics_v2['r2']:.3f}", f"{train_metrics_v3['r2']:.3f}"],
        textposition='inside',
        textfont=dict(size=10, color='white', family="Arial Black")
    ),
    row=1, col=2
)

fig_dashboard.add_trace(
    go.Bar(
        name='Test R²',
        x=['V1', 'V2', 'V3'],
        y=[test_metrics_v1['r2'], test_metrics_v2['r2'], test_metrics_v3['r2']],
        marker_color='#ff7f0e',
        showlegend=False,
        text=[f"{test_metrics_v1['r2']:.3f}", f"{test_metrics_v2['r2']:.3f}", f"{test_metrics_v3['r2']:.3f}"],
        textposition='inside',
        textfont=dict(size=10, color='white', family="Arial Black")
    ),
    row=1, col=2
)

# Accuracy Subplot
fig_dashboard.add_trace(
    go.Bar(
        name='Test Accuracy',
        x=['V1', 'V2', 'V3'],
        y=[test_metrics_v1['accuracy'], test_metrics_v2['accuracy'], test_metrics_v3['accuracy']],
        marker_color=['#1f77b4', '#ff7f0e', '#2ca02c'],
        showlegend=False,
        text=[f"{test_metrics_v1['accuracy']:.1f}%", f"{test_metrics_v2['accuracy']:.1f}%", f"{test_metrics_v3['accuracy']:.1f}%"],
        textposition='inside',
        textfont=dict(size=11, color='white', family="Arial Black")
    ),
    row=2, col=1
)

# Overfitting Subplot
fig_dashboard.add_trace(
    go.Bar(
        name='R² Difference',
        x=['V1', 'V2', 'V3'],
        y=overfitting_r2,
        marker_color='#9467bd',
        showlegend=False,
        text=[f"{x:.3f}" for x in overfitting_r2],
        textposition='inside',
        textfont=dict(size=11, color='white', family="Arial Black")
    ),
    row=2, col=2
)

fig_dashboard.update_layout(
    title=dict(
        text='<b>🏆 MODEL COMPARISON DASHBOARD: All Versions</b>',
        font=dict(size=24, family="Arial", color="#2c3e50"),
        x=0.5,
        y=0.98
    ),
    height=800,
    width=1200,
    template='plotly_white',
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        font=dict(size=12)
    )
)

# Update axes
fig_dashboard.update_xaxes(title_text="<b>Model</b>", row=1, col=1)
fig_dashboard.update_xaxes(title_text="<b>Model</b>", row=1, col=2)
fig_dashboard.update_xaxes(title_text="<b>Model</b>", row=2, col=1)
fig_dashboard.update_xaxes(title_text="<b>Model</b>", row=2, col=2)

fig_dashboard.update_yaxes(title_text="<b>MAE</b>", row=1, col=1)
fig_dashboard.update_yaxes(title_text="<b>R²</b>", row=1, col=2, range=[0, 1])
fig_dashboard.update_yaxes(title_text="<b>Accuracy (%)</b>", row=2, col=1, range=[0, 100])
fig_dashboard.update_yaxes(title_text="<b>R² Diff</b>", row=2, col=2)

fig_dashboard.show()

# ===== 6. SUMMARY TABLE =====
print("\n" + "="*70)
print("📋 MODEL COMPARISON SUMMARY TABLE")
print("="*70)

summary_data = {
    'Metric': ['Train MAE', 'Test MAE', 'Train R²', 'Test R²', 'Test Accuracy', 'Overfitting (R² Diff)'],
    'V1.0 Baseline': [
        f"{train_metrics_v1['mae']:,.0f}",
        f"{test_metrics_v1['mae']:,.0f}",
        f"{train_metrics_v1['r2']:.3f}",
        f"{test_metrics_v1['r2']:.3f}",
        f"{test_metrics_v1['accuracy']:.1f}%",
        f"{train_metrics_v1['r2'] - test_metrics_v1['r2']:.3f}"
    ],
    'V2.0 Weather': [
        f"{train_metrics_v2['mae']:,.0f}",
        f"{test_metrics_v2['mae']:,.0f}",
        f"{train_metrics_v2['r2']:.3f}",
        f"{test_metrics_v2['r2']:.3f}",
        f"{test_metrics_v2['accuracy']:.1f}%",
        f"{train_metrics_v2['r2'] - test_metrics_v2['r2']:.3f}"
    ],
    'V3.0 Optimized': [
        f"{train_metrics_v3['mae']:,.0f}",
        f"{test_metrics_v3['mae']:,.0f}",
        f"{train_metrics_v3['r2']:.3f}",
        f"{test_metrics_v3['r2']:.3f}",
        f"{test_metrics_v3['accuracy']:.1f}%",
        f"{train_metrics_v3['r2'] - test_metrics_v3['r2']:.3f}"
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n", summary_df.to_string(index=False))

# ===== 7. BEST MODEL IDENTIFICATION =====
print("\n" + "="*70)
print("🏆 BEST MODEL IDENTIFICATION")
print("="*70)

# Find best model based on test MAE
best_mae = min(test_metrics_v1['mae'], test_metrics_v2['mae'], test_metrics_v3['mae'])
best_mae_model = ['V1.0', 'V2.0', 'V3.0'][[test_metrics_v1['mae'], test_metrics_v2['mae'], test_metrics_v3['mae']].index(best_mae)]

# Find best model based on test R²
best_r2 = max(test_metrics_v1['r2'], test_metrics_v2['r2'], test_metrics_v3['r2'])
best_r2_model = ['V1.0', 'V2.0', 'V3.0'][[test_metrics_v1['r2'], test_metrics_v2['r2'], test_metrics_v3['r2']].index(best_r2)]

# Find best model based on test accuracy
best_acc = max(test_metrics_v1['accuracy'], test_metrics_v2['accuracy'], test_metrics_v3['accuracy'])
best_acc_model = ['V1.0', 'V2.0', 'V3.0'][[test_metrics_v1['accuracy'], test_metrics_v2['accuracy'], test_metrics_v3['accuracy']].index(best_acc)]

# Find best model based on lowest overfitting
best_gen = min(train_metrics_v1['r2'] - test_metrics_v1['r2'],
               train_metrics_v2['r2'] - test_metrics_v2['r2'],
               train_metrics_v3['r2'] - test_metrics_v3['r2'])
best_gen_model = ['V1.0', 'V2.0', 'V3.0'][[train_metrics_v1['r2'] - test_metrics_v1['r2'],
                                           train_metrics_v2['r2'] - test_metrics_v2['r2'],
                                           train_metrics_v3['r2'] - test_metrics_v3['r2']].index(best_gen)]

print(f"\n✅ Lowest Test MAE: {best_mae_model} ({best_mae:,.0f} arrivals)")
print(f"✅ Highest Test R²: {best_r2_model} ({best_r2:.3f})")
print(f"✅ Highest Test Accuracy: {best_acc_model} ({best_acc:.1f}%)")
print(f"✅ Best Generalization (lowest overfitting): {best_gen_model} ({best_gen:.3f} R² diff)")

# Overall champion (majority vote or best accuracy)
if best_acc_model == best_r2_model == best_mae_model:
    champion = best_acc_model
else:
    # Default to best accuracy as champion
    champion = best_acc_model

print(f"\n🏆 OVERALL CHAMPION MODEL: {champion}")
print("="*70)

In [ ]:
future_dates = pd.date_range(start='2026-01-01', end='2026-12-01', freq='MS')
future_df = pd.DataFrame(index=future_dates)
future_df.index.name = 'Month'

# Conservative estimates for 2026
future_df['Peak_Season'] = [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1]
future_df['Philippine_Holidays'] = [2, 1, 2, 3, 2, 2, 1, 2, 1, 1, 3, 4]
future_df['Top10Market_Holidays'] = [26, 19, 16, 23, 27, 17, 10, 12, 17, 22, 20, 27]
future_df['Avg_HighTemp'] = [29, 29, 30, 31, 32, 32, 32, 32, 32, 31, 30, 29]
future_df['Avg_LowTemp'] = [25, 25, 25, 26, 26, 26, 26, 26, 26, 26, 25, 25]
future_df['Precipitation'] = [0.7, 0.5, 0.4, 0.3, 0.4, 0.7, 0.9, 0.8, 0.9, 1.0, 0.9, 0.7]
future_df['Inflation_Rate'] = [3.0, 3.0, 2.9, 2.9, 2.8, 2.8, 2.7, 2.7, 2.6, 2.6, 2.5, 2.5]
future_df['is_December'] = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
future_df['is_Lockdown'] = [0] * 12

# Add time features
future_df['Year'] = future_df.index.year
future_df['Month_Num'] = future_df.index.month

# For lag features, use last known values
last_values = df_v3['Arrivals'].iloc[-2:].values
future_df['Lag_1'] = df_v3['Arrivals'].iloc[-1]
future_df['Lag_2'] = df_v3['Arrivals'].iloc[-2] if len(last_values) > 1 else df_v3['Arrivals'].iloc[-1]
future_df['Rolling_Mean_3m'] = df_v3['Arrivals'].iloc[-3:].mean()

# Growth rate (using recent trend)
future_df['Growth_Rate'] = ((future_df['Lag_1'] - future_df['Lag_2']) / future_df['Lag_2'] * 100).fillna(0)
future_df['Sudden_Increase'] = 0  # Conservative assumption

# Make predictions
future_X = future_df[features_v3]
future_pred = model_v3.predict(future_X)
future_pred_rounded = np.round(future_pred).astype(int)
future_df['Predicted_Arrivals'] = future_pred_rounded

print("\n📅 Predicted Arrivals for 2026:")
print(future_df[['Predicted_Arrivals']].to_string())

# Plot future predictions
fig_future = go.Figure()

# Historical data (last 3 years)
historical = df_v3[df_v3.index >= '2023-01-01']
fig_future.add_trace(go.Scatter(
    x=historical.index, y=historical['Arrivals'],
    mode='lines+markers', name='Historical',
    line=dict(color='blue')
))

# Future predictions
fig_future.add_trace(go.Scatter(
    x=future_df.index, y=future_df['Predicted_Arrivals'],
    mode='lines+markers', name='Predicted 2026',
    line=dict(color='red', dash='dash'),
    marker=dict(size=8)
))

fig_future.update_layout(
    title="Tourist Arrivals: Historical (2023-2025) and Predicted (2026)",
    xaxis_title="Date",
    yaxis_title="Arrivals",
    hovermode='x unified',
    template='plotly_white',
    height=500
)
fig_future.show()

